<a href="https://colab.research.google.com/github/AnthonyMath1022/hugo-book/blob/main/N_linear_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import yfinance as yf
import datetime as dt
import torch
import sklearn.metrics as metrics
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_style('whitegrid')

torch.manual_seed(123456)
np.random.seed(123456)
'''
from tsl.datasets import PemsBay
original_data = PemsBay().load()

data = original_data[0]
num_nodes = 50
data = data.iloc[:, :num_nodes]
'''
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/traffic/

data = np.load('one_intersection.npy')
num_nodes = data.shape[1]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/traffic


In [ ]:
print(data.shape)

(52128, 20)


In [ ]:
  # use latest 10,000 time ste
total_len = len(data)  # 52128

train_len = int(total_len * 0.70)  # 36489
val_len   = int(total_len * 0.20)  # 10425
test_len  = total_len - train_len - val_len  # 5214

train_df = data[:train_len]
val_df   = data[train_len:train_len + val_len]
test_df  = data[train_len + val_len:]



feat_scaler = MinMaxScaler()
feat_scaler.fit(train_df)          # fit *once* on raw training values

train_scaled = feat_scaler.transform(train_df)
val_scaled   = feat_scaler.transform(val_df)
test_scaled  = feat_scaler.transform(test_df)

In [ ]:
total = data.shape[1]
print(f"Train: {len(train_df)} ({len(train_df)/total*100:.1f}%)")
print(f"Val:   {len(val_df)}   ({len(val_df)/total*100:.1f}%)")
print(f"Test:  {len(test_df)}  ({len(test_df)/total*100:.1f}%)")
print(f"Total: {total}")

Train: 36489 (182445.0%)
Val:   10425   (52125.0%)
Test:  5214  (26070.0%)
Total: 20


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class Model(nn.Module):
    """
    Normalization-Linear
    """
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len

        # Use this line if you want to visualize the weights
        # self.Linear.weight = nn.Parameter((1/self.seq_len)*torch.ones([self.pred_len,self.seq_len]))
        self.channels = configs.enc_in
        self.individual = configs.individual
        if self.individual:
            self.Linear = nn.ModuleList()
            for i in range(self.channels):
                self.Linear.append(nn.Linear(self.seq_len,self.pred_len))
        else:
            self.Linear = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x):
        # x: [Batch, Input length, Channel]
        seq_last = x[:,-1:,:].detach()
        x = x - seq_last
        if self.individual:
            output = torch.zeros([x.size(0),self.pred_len,x.size(2)],dtype=x.dtype).to(x.device)
            for i in range(self.channels):
                output[:,:,i] = self.Linear[i](x[:,:,i])
            x = output
        else:
            x = self.Linear(x.permute(0,2,1)).permute(0,2,1)
        x = x + seq_last
        return x # [Batch, Output length, Channel]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_length, pred_length=1):
        """
        data: numpy array of shape (n_samples, n_features)
        seq_length: length of input sequence (lookback window)
        pred_length: length of prediction horizon
        """
        self.data = torch.FloatTensor(data)
        self.seq_length = seq_length
        self.pred_length = pred_length

    def __len__(self):
        return len(self.data) - self.seq_length - self.pred_length + 1

    def __getitem__(self, idx):
        # Input sequence (x): from idx to idx+seq_length
        x = self.data[idx:idx+self.seq_length, :]

        # Target (y): the next 'pred_length' values (all columns)
        y = self.data[idx+self.seq_length:idx+self.seq_length+self.pred_length, :]

        return x, y



In [ ]:
from types import SimpleNamespace
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

configs = SimpleNamespace(
    seq_len=168,
    pred_len=1,
    enc_in=num_nodes,   # IMPORTANT: number of nodes/features (20)
    individual=True
)

train_dataset = TimeSeriesDataset(train_scaled, configs.seq_len, configs.pred_len)

val_dataset = TimeSeriesDataset(val_scaled, configs.seq_len, configs.pred_len)

test_dataset = TimeSeriesDataset(test_scaled, configs.seq_len, configs.pred_len)



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = Model(configs)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001,weight_decay=1e-5)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)
criterion = torch.nn.MSELoss()

# 2. OPTIMIZATION: Add Early Stopping Logic
best_val_loss = float('inf')
patience = 20
trigger_times = 0

train_epoch_losses = []
test_losses = []
val_epoch_losses = []

best_val_loss = float('inf')
patience = 20
trigger_times = 0

train_epoch_losses = []
test_losses = []
val_epoch_losses = []

print("Starting Training...")
print("Starting Training...")
for epoch in range(200): # Max epochs
    # --- TRAIN ---
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x, y = x.float(), y.float()

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_epoch_losses.append(train_loss)

    # --- VAL ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.float(), y.float()
            pred = model(x)
            loss = criterion(pred, y)
            val_loss += loss.item()
    val_loss /= len(val_loader)
    val_epoch_losses.append(val_loss)
    scheduler.step(val_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")

    # --- EARLY STOPPING ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trigger_times = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load('best_model.pth'))
model.eval()

predictions, actuals = [], []
test_loss, num_batches = 0.0, 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.float(), y.float()
        pred = model(x)

        # Accumulate test loss
        test_loss += criterion(pred, y).item()
        num_batches += 1

        # Reshape & inverse transform
        pred_np = pred.cpu().numpy().reshape(-1, data.shape[1])
        y_np    = y.cpu().numpy().reshape(-1, data.shape[1])

        predictions.append(feat_scaler.inverse_transform(pred_np))
        actuals.append(feat_scaler.inverse_transform(y_np))

print(f"Test Loss (Scaled): {test_loss / num_batches:.6f}")

all_preds = np.concatenate(predictions, axis=0)
all_actuals = np.concatenate(actuals, axis=0)

# Compute 1-step ahead metrics only (clean & standard)
step0_mask = slice(None, None, configs.pred_len)
p1 = all_preds[step0_mask]
a1 = all_actuals[step0_mask]

rmse_1step = np.sqrt(np.mean((p1 - a1)**2))
print(f"1-Step Test RMSE (Original Scale): {rmse_1step:.4f}")

Starting Training...
Starting Training...
Epoch 0: Train Loss = 0.002223, Val Loss = 0.000989
Epoch 10: Train Loss = 0.000711, Val Loss = 0.000718
Epoch 20: Train Loss = 0.000690, Val Loss = 0.000694
Epoch 30: Train Loss = 0.000681, Val Loss = 0.000685
Epoch 40: Train Loss = 0.000673, Val Loss = 0.000680
Epoch 50: Train Loss = 0.000670, Val Loss = 0.000680
Epoch 60: Train Loss = 0.000667, Val Loss = 0.000676
Epoch 70: Train Loss = 0.000665, Val Loss = 0.000675
Epoch 80: Train Loss = 0.000664, Val Loss = 0.000674
Epoch 90: Train Loss = 0.000663, Val Loss = 0.000674
Epoch 100: Train Loss = 0.000663, Val Loss = 0.000674
Early stopping at epoch 106
Test Loss (Scaled): 0.000926
1-Step Test RMSE (Original Scale): 2.1558


In [ ]:
# Combine all predictions and actuals
all_preds = np.concatenate(predictions, axis=0)
all_actuals = np.concatenate(actuals, axis=0)

# Calculate RMSE and MAE in original scale
rmse = np.sqrt(np.mean((all_preds - all_actuals) ** 2))
mae = np.mean(np.abs(all_preds - all_actuals))

print(f"\n=== Test Results (Original Scale) ===")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Average Price Level: {np.mean(all_actuals):.2f}")
print(f"RMSE as % of Avg Price: {(rmse/np.mean(all_actuals)*100):.2f}%")


# Plot predictions vs actuals for first stock
# Take only step-0 prediction from each window → shape (214, 9)
all_preds_1step   = all_preds[0::configs.pred_len]
all_actuals_1step = all_actuals[0::configs.pred_len]
'''
for i, col in enumerate(data.columns):
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(all_actuals_1step[:, i], label='Actual',    color='blue', alpha=0.8)
    ax.plot(all_preds_1step[:, i],   label='Predicted', color='red',  alpha=0.7, linestyle='--')
    ax.set_title(f'{col} — NLinear-Closing Price')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Price')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    #plt.savefig(f'{col} - NLinear Closing Price.png')
    plt.show()
'''


=== Test Results (Original Scale) ===
RMSE: 2.1558
MAE: 1.0925
Average Price Level: 61.50
RMSE as % of Avg Price: 3.51%


"\nfor i, col in enumerate(data.columns):\n    fig, ax = plt.subplots(figsize=(14, 4))\n    ax.plot(all_actuals_1step[:, i], label='Actual',    color='blue', alpha=0.8)\n    ax.plot(all_preds_1step[:, i],   label='Predicted', color='red',  alpha=0.7, linestyle='--')\n    ax.set_title(f'{col} — NLinear-Closing Price')\n    ax.set_xlabel('Trading Day')\n    ax.set_ylabel('Price')\n    ax.legend()\n    ax.grid(True, alpha=0.3)\n    plt.tight_layout()\n    #plt.savefig(f'{col} - NLinear Closing Price.png')\n    plt.show()\n"

In [ ]:

from sklearn import metrics
def mean_absolute_percentage_error(y_test, y_pred):
    y_true, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

def mean_squared_prediction_error(y_test, y_pred):
    y_true, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(((y_test - y_pred))**2)


for i in range(data.shape[1]):
    print(f"\nMetrics for Feature {i+1}:") # Changed {col} to {i+1}
    print(f'MSE: {metrics.mean_squared_error(all_actuals[:, i], all_preds[:, i])}')
    print(f'MAE: {metrics.mean_absolute_error(all_actuals[:, i], all_preds[:, i])}')
    print(f'RMSE: {np.sqrt(metrics.mean_squared_error(all_actuals[:, i], all_preds[:, i]))}')
    print(f'MAPE: {mean_absolute_percentage_error(all_actuals[:, i], all_preds[:, i])}')
    print(f'MSPE: {mean_squared_prediction_error(all_actuals[:, i], all_preds[:, i])}')
    print(f'sqrt MSPE: {np.sqrt(mean_squared_prediction_error(all_actuals[:, i], all_preds[:, i]))}')
    print(f'R2: {r2_score(all_actuals[:, i], all_preds[:, i])}')

# Overall metrics
print("\nOverall Metrics:")
print(f'MSE: {metrics.mean_squared_error(all_actuals, all_preds)}')
print(f'MAE: {metrics.mean_absolute_error(all_actuals, all_preds)}')
print(f'RMSE: {np.sqrt(metrics.mean_squared_error(all_actuals, all_preds))}')
print(f'MAPE: {mean_absolute_percentage_error(all_actuals, all_preds)}')
print(f'MSPE: {mean_squared_prediction_error(all_actuals, all_preds)}')
print(f'sqrt MSPE: {np.sqrt(mean_squared_prediction_error(all_actuals, all_preds))}')
print(f'R2: {r2_score(all_actuals, all_preds)}')


Metrics for Feature 1:
MSE: 4.891507625579834
MAE: 0.9868971109390259
RMSE: 2.211675298406128
MAPE: inf
MSPE: 4.891507625579834
sqrt MSPE: 2.2116754055023193
R2: 0.9171711206436157

Metrics for Feature 2:
MSE: 4.817555904388428
MAE: 1.425100326538086
RMSE: 2.194893141906555
MAPE: 3.0604324340820312
MSPE: 4.817555904388428
sqrt MSPE: 2.1948931217193604
R2: 0.9605593681335449

Metrics for Feature 3:
MSE: 3.7485852241516113
MAE: 1.2114335298538208
RMSE: 1.9361263450900128
MAPE: 2.3503711223602295
MSPE: 3.7485852241516113
sqrt MSPE: 1.9361263513565063
R2: 0.9737730026245117

Metrics for Feature 4:
MSE: 7.157501220703125
MAE: 1.1804803609848022
RMSE: 2.67535067247326
MAPE: inf
MSPE: 7.157501220703125
sqrt MSPE: 2.6753506660461426
R2: 0.9491510987281799

Metrics for Feature 5:
MSE: 0.1734551340341568
MAE: 0.24781496822834015
RMSE: 0.41647945211517556
MAPE: 0.42388463020324707
MSPE: 0.1734551340341568
sqrt MSPE: 0.4164794385433197
R2: 0.9950305819511414

Metrics for Feature 6:
MSE: 6.8982267

In [ ]:
# After training, use the best model for final in-sample predictions
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

final_train_actuals = []
final_train_preds = []

with torch.no_grad():
    for x, y in train_loader:
        x, y = x.float(), y.float()
        pred = model(x)

        final_train_actuals.append(y.cpu().numpy())
        final_train_preds.append(pred.cpu().numpy())

# Concatenate all batches
train_actuals = np.concatenate(final_train_actuals, axis=0)
train_predictions = np.concatenate(final_train_preds, axis=0)

# Reshape to 2D for inverse_transform
if train_actuals.ndim == 3:
    train_actuals = train_actuals.reshape(-1, train_actuals.shape[-1])

if train_predictions.ndim == 3:
    train_predictions = train_predictions.reshape(-1, train_predictions.shape[-1])

# Inverse transform back to original price scale
train_actuals_original = feat_scaler.inverse_transform(train_actuals)
train_predictions_original = feat_scaler.inverse_transform(train_predictions)

# Overall in-sample MSE and RMSE
mse = metrics.mean_squared_error(train_actuals_original, train_predictions_original)
rmse = np.sqrt(mse)

print(f"In-sample MSE is : {mse}")
print(f"In-sample RMSE is : {rmse}")

# Per asset in-sample MSE and RMSE
for i in range(data.shape[1]):
    mse_i = metrics.mean_squared_error(
        train_actuals_original[:, i],
        train_predictions_original[:, i]
    )

    rmse_i = np.sqrt(mse_i)

    print(f"\nMetrics for {col}:")
    print(f"MSE: {mse_i}")
    print(f"RMSE: {rmse_i}")

In-sample MSE is : 3.221484661102295
In-sample RMSE is : 1.7948494814614107


NameError: name 'col' is not defined

In [ ]:
from google.colab import runtime
runtime.unassign()